In [ ]:
# of session completion given context and template choice. We then use 
# this model as a greedy policy to select the template with highest 
# predicted reward from the eligible pool. Since we only observe 
# rewards for the logged template, we evaluate the new policy using 
# inverse propensity scoring (IPS), assuming the logging policy was 
# uniform random within pools.

import polars as pl
import numpy as np

In [ ]:
TRAIN_PATH = "train_rsample_100k.parquet"  # your good sample
VAL_PATH   = "val_rsample_100k.parquet"

train = pl.read_parquet(TRAIN_PATH)
val   = pl.read_parquet(VAL_PATH)

print("Train:", train.shape)
print("Val:", val.shape)
print("Train baseline:", train.select(pl.mean("session_end_completed")).item())
print("Val baseline:", val.select(pl.mean("session_end_completed")).item())

Train: (100000, 10)
Val: (100000, 10)
Train baseline: 0.14704
Val baseline: 0.13557


In [ ]:
train.head(10)

datetime,ui_language,selected_template,eligible_templates,history,history_length,n_eligible,time_of_day,session_end_completed,hour_utc
f64,str,str,list[str],list[struct[2]],u32,u32,f64,bool,f64
9.28338,"""fr""","""L""","[""K"", ""H"", … ""D""]","[{""J"",5.882937}, {""A"",4.013899}, {""L"",0.999994}]",3,9,9.28338,false,6.801111
9.623044,"""es""","""D""","[""K"", ""H"", … ""D""]","[{""A"",23.989475}, {""E"",22.989477}, … {""E"",0.999996}]",11,9,9.623044,false,14.953056
5.172986,"""de""","""K""","[""G"", ""E"", … ""D""]","[{""E"",4.50845}, {""G"",3.508451}, {""G"",1.693507}]",3,10,5.172986,false,4.151667
6.269595,"""en""","""G""","[""G"", ""E"", … ""D""]","[{""K"",8.594528}, {""F"",7.468275}, … {""G"",1.001365}]",7,9,6.269595,false,6.470278
0.913009,"""es""","""L""","[""G"", ""E"", … ""D""]","[{""L"",0.999999}]",1,9,0.913009,false,21.912222
0.636458,"""es""","""D""","[""G"", ""E"", … ""D""]","[{""A"",28.029877}, {""G"",25.972994}, … {""D"",1.010333}]",14,10,0.636458,false,15.275
9.939329,"""es""","""F""","[""K"", ""H"", … ""A""]","[{""H"",6.041422}, {""F"",5.041424}, … {""J"",1.026598}]",6,10,9.939329,false,22.543889
5.212697,"""en""","""E""","[""G"", ""E"", … ""D""]","[{""C"",5.810475}, {""H"",4.810474}, … {""E"",0.999998}]",5,9,5.212697,false,5.104722
5.104537,"""fr""","""A""","[""G"", ""E"", … ""D""]","[{""B"",29.4512}, {""A"",27.476376}, … {""J"",0.438472}]",34,10,5.104537,false,2.508889


In [ ]:
train = train.with_columns(
    ((pl.col("datetime") % 1) * 24)
        .floor()
        .cast(pl.Int8)
        .alias("hour_utc")
)

val = val.with_columns(
    ((pl.col("datetime") % 1) * 24)
        .floor()
        .cast(pl.Int8)
        .alias("hour_utc")
)

In [ ]:
#! Don't forget to drop one for Logistic regression (perfect multicollinearity)

In [ ]:
train.select("ui_language").unique()

ui_language
str
"""pl"""
"""zs"""
"""el"""
"""ja"""
"""es"""
…
"""tr"""
"""ko"""
"""hu"""


In [ ]:
val.select("ui_language").unique()

ui_language
str
"""ro"""
"""zs"""
"""hi"""
"""pl"""
"""dn"""
…
"""ko"""
"""hu"""
"""el"""


In [ ]:
train = train.to_dummies(columns=["ui_language"])
val   = val.to_dummies(columns=["ui_language"])

In [ ]:
print(len(train.columns))
train.head(10)

32


datetime,ui_language_ar,ui_language_cs,ui_language_de,ui_language_dn,ui_language_el,ui_language_en,ui_language_es,ui_language_fr,ui_language_hi,ui_language_hu,ui_language_id,ui_language_it,ui_language_ja,ui_language_ko,ui_language_pl,ui_language_pt,ui_language_ro,ui_language_ru,ui_language_th,ui_language_tr,ui_language_uk,ui_language_vi,ui_language_zs,selected_template,eligible_templates,history,history_length,n_eligible,time_of_day,session_end_completed,hour_utc
f64,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,u8,str,list[str],list[struct[2]],u32,u32,f64,bool,i8
9.28338,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""L""","[""K"", ""H"", … ""D""]","[{""J"",5.882937}, {""A"",4.013899}, {""L"",0.999994}]",3,9,9.28338,false,6
9.623044,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""D""","[""K"", ""H"", … ""D""]","[{""A"",23.989475}, {""E"",22.989477}, … {""E"",0.999996}]",11,9,9.623044,false,14
5.172986,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""K""","[""G"", ""E"", … ""D""]","[{""E"",4.50845}, {""G"",3.508451}, {""G"",1.693507}]",3,10,5.172986,false,4
6.269595,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""G""","[""G"", ""E"", … ""D""]","[{""K"",8.594528}, {""F"",7.468275}, … {""G"",1.001365}]",7,9,6.269595,false,6
0.913009,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""L""","[""G"", ""E"", … ""D""]","[{""L"",0.999999}]",1,9,0.913009,false,21
0.636458,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""D""","[""G"", ""E"", … ""D""]","[{""A"",28.029877}, {""G"",25.972994}, … {""D"",1.010333}]",14,10,0.636458,false,15
9.939329,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""F""","[""K"", ""H"", … ""A""]","[{""H"",6.041422}, {""F"",5.041424}, … {""J"",1.026598}]",6,10,9.939329,false,22
5.212697,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""E""","[""G"", ""E"", … ""D""]","[{""C"",5.810475}, {""H"",4.810474}, … {""E"",0.999998}]",5,9,5.212697,false,5
5.104537,0,0,0,0,0,0,0,1,0,0,0,0,0,0,0,0,0,0,0,0,0,0,0,"""A""","[""G"", ""E"", … ""D""]","[{""B"",29.4512}, {""A"",27.476376}, … {""J"",0.438472}]",34,10,5.104537,false,2


In [ ]:
# same order in both val and train
set(train.columns) == set(val.columns)

assert train.columns == val.columns



## To drop for modeling: history

## history_length: number of past exposures
## n_eligible: number of templates a user is able to receive at the moment

## Add from history: length between template sent

In [ ]:
#                       their most recent notification (any template).


train = train.with_columns(
    pl.col("history").map_elements(
        lambda hist: 100.0 if (hist is None or len(hist) == 0)
        else float(min(item["n_days"] for item in hist)),
        return_dtype=pl.Float64
    ).alias("time_since_last_notification_days")
)

val = val.with_columns(
    pl.col("history").map_elements(
        lambda hist: 100.0 if (hist is None or len(hist) == 0)
        else float(min(item["n_days"] for item in hist)),
        return_dtype=pl.Float64
    ).alias("time_since_last_notification_days")
)

In [ ]:
from collections import Counter

train = train.with_columns(
    pl.col("history").map_elements(
        lambda hist: 0 if (hist is None or len(hist) == 0)
        else max(
            Counter(
                # take 5 most recent by smallest n_days
                [item["template"] for item in sorted(hist, key=lambda d: d["n_days"])[:5]]
            ).values()
        ),
        return_dtype=pl.Int16
    ).alias("max_template_count_last_5")
)

val = val.with_columns(
    pl.col("history").map_elements(
        lambda hist: 0 if (hist is None or len(hist) == 0)
        else max(
            Counter(
                [item["template"] for item in sorted(hist, key=lambda d: d["n_days"])[:5]]
            ).values()
        ),
        return_dtype=pl.Int16
    ).alias("max_template_count_last_5")
)

In [ ]:
## Add day index 
train = train.with_columns(
    (pl.col("datetime").floor().cast(pl.Int32) + 1).alias("day_index")
)

val = val.with_columns(
    (pl.col("datetime").floor().cast(pl.Int32) + 1).alias("day_index")
)

In [ ]:
train = train.drop(["history"])
val   = val.drop(["history"])

In [ ]:
OUT_TRAIN = "train_preprocessed_100k.parquet"
OUT_VAL   = "val_preprocessed_100k.parquet"

train.write_parquet(OUT_TRAIN)
val.write_parquet(OUT_VAL)

print("Wrote:", OUT_TRAIN, OUT_VAL)

Wrote: train_preprocessed_100k.parquet val_preprocessed_100k.parquet
